# 01 pyproj Reprojection Warm-up

**Goal:** get comfortable reading a CRS definition and calling a transform directly, instead of trusting geopandas to handle it invisibly.

**Exercise:** pull two Colorado datasets in different native projections (ex. a SNOTEL station list in lat/lon, a county boundary layer in a projected CRS) and reproject one to match the other by hand before doing anything else with them.

**Why this matters:** a silent CRS mismatch is one of the easiest ways to misalign two otherwise-correct datasets by a few hundred meters without any error being raised.

In [ ]:
import pandas as pd
import geopandas as gpd
import pyproj

In [ ]:
# Load SNOTEL station data
snotel = pd.read_csv("data/snotel_stations.csv")

# Convert station coordinates into a GeoDataFrame
snotel_gdf = gpd.GeoDataFrame(
    snotel,
    geometry=gpd.points_from_xy(
        snotel.longitude,
        snotel.latitude
    ),
    crs="EPSG:4326"   # WGS84 latitude/longitude
)

snotel_gdf.head()

In [ ]:
# Check the CRS
snotel_gdf.crs

In [ ]:
counties = gpd.read_file("data/colorado_counties.shp")
# check CRS
counties.crs

In [ ]:
# Define the source and target CRS for your two datasets
source_crs = pyproj.CRS("EPSG:4326")   # WGS84 latitude/longitude
target_crs = pyproj.CRS("EPSG:26913")  # NAD83/UTM zone 13N (Colorado)


In [ ]:
transformer = pyproj.Transformer.from_crs(
    source_crs,
    target_crs,
    always_xy=True
)

x, y = transformer.transform(
    snotel_gdf.geometry.x,
    snotel_gdf.geometry.y
)

snotel_gdf["easting"] = x
snotel_gdf["northing"] = y

**Check yourself:** print the same station's coordinates in both CRSs and confirm you can explain, in one sentence, why the numbers look so different even though it's the same physical point.

In [ ]:
# Final check
print("SNOTEL CRS:", snotel_gdf.crs)
print("County CRS:", counties.crs)